# Fase 2A — Machine Learning: LR, RF y XGBoost (IroSvA) — v2

**Qué es la v2.** Es la **continuación de la v1** (`ML_TFIDF_10k_GroupNestedCV_5x3_CORREGIDO.ipynb`), no un experimento nuevo. Usa el mismo código de la v1 (carga, grupos, TF-IDF de 10 000 términos + 9 rasgos lingüísticos, pipelines de LR/RF/XGBoost) y **lee** de la carpeta v1 la configuración final de cada modelo (`seleccion_final_ml.csv`: preprocesamiento, n-gramas e hiperparámetros elegidos solo con el train). Agrega:

1. **5 semillas** (42, 52, 62, 72, 82), las mismas de los Transformers: cada modelo final se reentrena con todo el train y cada semilla, **sin cambiar ningún hiperparámetro**, y se reporta media ± desviación en el test. La Regresión Logística (`liblinear`) es determinista, por lo que sus 5 resultados deberían ser idénticos (se comprueba).
2. **Verificación**: la semilla 42 debe reproducir exactamente las predicciones de la v1.
3. **Interpretabilidad con SHAP** para LR, RF y XGBoost: palabras y rasgos lingüísticos que más influyen por variante.

**Lo que NO hace:** no repite la Nested CV ni la búsqueda final (los resultados de desarrollo siguen siendo los de la v1). El test sigue bloqueado (`EJECUTAR_TEST_OFICIAL`); al habilitarlo solo se miden las 5 semillas, sin elegir nada. Solo CPU; unos minutos. Requiere `shap` (`py -m uv add shap`) para la sección 11. Resultados en `../data/resultados_ml_groupcv_tfidf_lr_rf_10k_v2/`.

## 1. Librerías y entorno

Importaciones, stopwords NLTK y versiones de los paquetes utilizados.

In [ ]:
import ast
import json
import time
import html
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import optuna
import sklearn
import xgboost
import nltk
from nltk.corpus import stopwords
from IPython.display import display, HTML, Image

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_score
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)
from xgboost import XGBClassifier

# Lista original del notebook LR/RF. Se usa la misma para los tres modelos.
try:
    STOP_WORDS = stopwords.words("spanish")
except LookupError:
    if not nltk.download("stopwords", quiet=True):
        raise RuntimeError(
            "Instala el corpus NLTK de stopwords para reproducir el TF-IDF original."
        )
    STOP_WORDS = stopwords.words("spanish")

optuna.logging.set_verbosity(optuna.logging.WARNING)

VERSIONES = {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "optuna": optuna.__version__,
}
print("Versiones:", " | ".join(f"{k} {v}" for k, v in VERSIONES.items()))

import matplotlib
import matplotlib.pyplot as plt

## 2. Configuración

Igual que la v1 (rutas, protocolo y panel de seguimiento), con la carpeta de salida `_v2`, las 5 semillas y los parámetros de interpretabilidad.

In [ ]:
DATA_DIR = Path("../data")
RANDOM_STATE = 42
MODELOS_LR_RF = ["LR", "RF"]
PREPROCESAMIENTOS = {"normal": "", "stem": "_stem", "lemma": "_lemma"}
FEATURE_COLS = ["n_exc", "n_int", "n_may", "n_emo", "n_ris", "n_neg", "n_elo", "n_com", "n_pun"]

# ------------------------------------------------------------
# "prueba" valida rápidamente todo el pipeline.
# "completo" = experimento definitivo 5×3.
# ------------------------------------------------------------
MODO = "completo"  # v2: siempre completo (lee la selección final de la v1)

if MODO == "prueba":
    VARIANTES = ["mx"]
    N_OUTER = 2
    N_INNER = 2
    N_TRIALS_NESTED = 2
    N_TRIALS_FINAL = 2
elif MODO == "completo":
    VARIANTES = ["mx", "es", "cu"]
    N_OUTER = 5
    N_INNER = 3
    N_TRIALS_NESTED = 50
    N_TRIALS_FINAL = 50
else:
    raise ValueError("MODO debe ser 'prueba' o 'completo'.")

# Los resultados de prueba quedan aislados en su propia subcarpeta.
# v2: la v1 se LEE (selección final y Nested CV); la v2 escribe en su propia carpeta.
OUT_DIR_V1 = DATA_DIR / "resultados_ml_groupcv_tfidf_lr_rf_10k"
OUT_DIR = DATA_DIR / "resultados_ml_groupcv_tfidf_lr_rf_10k_v2"
if MODO == "prueba":
    OUT_DIR = OUT_DIR / "prueba"
MODELS_DIR = OUT_DIR / "modelos_finales"
for carpeta in [OUT_DIR, MODELS_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

N_JOBS_GRID = -1  # Paralelismo de GridSearchCV para LR/RF
N_JOBS_XGB = -1  # Dentro de XGBoost; CV de Optuna secuencial

# Esta corrida fija el PERFIL ORIGINAL de LR/RF en los tres algoritmos.
# Los n-gramas se escogen INTERNAMENTE (1,1) vs (1,2), no con Outer Test.
NGRAMAS_OPCIONES = {"uni": (1, 1), "uni_bi": (1, 2)}
TFIDF_PERFIL = "LR_RF_ORIGINAL_10K_STOPWORDS_ES_NGRAMAS_INNER"

# Test oficial: solo con el protocolo congelado y en modo completo.
EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""

# v2 — mismas 5 semillas que los Transformers e interpretabilidad
SEMILLAS_FINALES = [42, 52, 62, 72, 82]
MODELOS_ML = ["LR", "RF", "XGBoost"]
MIN_TEXTOS_PALABRA = 5  # una palabra entra al análisis global si aparece en ≥ 5 tuits del test
TOP_PALABRAS = 8  # palabras por sentido (irónico / no irónico) en la figura global

OOF_REGISTROS = []  # Predicciones independientes del Outer Test (sin datos oficiales de test)

print(f"[CONFIG][ML] modo={MODO} | variantes={VARIANTES} | algoritmos=LR, RF, XGBoost")
print(
    f"[CONFIG][ML] Nested={N_OUTER}×{N_INNER} | XGBoost trials={N_TRIALS_NESTED} por Outer (final={N_TRIALS_FINAL})"
)
print(f"[CONFIG][ML] TF-IDF={TFIDF_PERFIL} | n-gramas={NGRAMAS_OPCIONES}")
print(f"[CONFIG][ML] stopwords NLTK={len(STOP_WORDS)} | test oficial={EJECUTAR_TEST_OFICIAL}")
print(f"[CONFIG][ML] OUT_DIR={OUT_DIR.resolve()}")

# Seguimiento: panel de progreso común a todos los notebooks de modelos.
NOMBRE_EXPERIMENTO = "ML v2"
HEARTBEAT_SEGUNDOS = 30
# Pitido de Windows al terminar o detenerse una etapa larga (False para desactivarlo).
AVISO_SONORO = True


class Monitor:
    """Panel único de progreso + monitor.log, con estado activo / terminado / error.

    Clase común a todos los notebooks de modelos (ML, Hard Voting y Transformers).
    """

    def __init__(self, ruta, total, ya_completados=0, unidad="Fine-tunings completados"):
        self.ruta = Path(ruta)
        self.ruta.mkdir(parents=True, exist_ok=True)
        self.log = self.ruta / "monitor.log"
        self.total = int(total)
        self.unidad = unidad
        self.completados = int(ya_completados)
        self.completados_inicio = int(ya_completados)
        self.inicio = time.monotonic()
        self.ultimo_pulso = 0
        self.etapa = "Inicializando"
        self.estado = "activo"  # activo | terminado | error
        self.handle = display(HTML(f"<b>Preparando {NOMBRE_EXPERIMENTO}...</b>"), display_id=True)
        self.mostrar(forzar=True)

    def registrar(self, msg):
        with self.log.open("a", encoding="utf-8") as f:
            f.write(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}\n")

    def mostrar(self, etapa=None, forzar=False):
        if etapa is not None:
            self.etapa = str(etapa)
        ahora = time.monotonic()
        if not forzar and ahora - self.ultimo_pulso < HEARTBEAT_SEGUNDOS:
            return
        self.ultimo_pulso = ahora
        minutos = (ahora - self.inicio) / 60
        hechos_sesion = self.completados - self.completados_inicio
        faltan = max(0, self.total - self.completados)
        eta = (
            f"~{minutos / hechos_sesion * faltan:.0f} min"
            if hechos_sesion >= 2
            else "calculando..."
        )
        marca = datetime.now().strftime("%H:%M:%S")
        color, titulo = {
            "activo": ("#777", f"{NOMBRE_EXPERIMENTO} — seguimiento en vivo"),
            "terminado": ("#2e7d32", f"✅ {NOMBRE_EXPERIMENTO} — TERMINADO"),
            "error": ("#c62828", f"❌ {NOMBRE_EXPERIMENTO} — DETENIDO"),
        }[self.estado]
        grosor = 1 if self.estado == "activo" else 3
        tiempo = (
            f"Tiempo: {minutos:.1f} min | ETA orientativo: {eta}"
            if self.estado == "activo"
            else f"Duración total: {minutos:.1f} min | Finalizó a las {marca}"
        )
        panel = (
            f'<div style="font-family:system-ui;padding:12px;border:{grosor}px solid {color};'
            'border-radius:8px;max-width:800px">'
            f'<b style="color:{color}">{titulo}</b><br>'
            f"Etapa: <b>{html.escape(self.etapa)}</b><br>"
            f"{self.unidad}: {self.completados}/{self.total} "
            f"({100*self.completados/max(1,self.total):.1f}%)<br>"
            f'<progress value="{self.completados}" max="{max(1,self.total)}" '
            'style="width:100%"></progress><br>'
            f"{tiempo}<br>"
            f"Última señal: {marca}<br>Log: {html.escape(str(self.log))}</div>"
        )
        self.handle.update(HTML(panel))

    def iniciar(self, msg):
        self.registrar(msg)
        self.mostrar(etapa=msg, forzar=True)

    def pulso(self):
        if time.monotonic() - self.ultimo_pulso >= HEARTBEAT_SEGUNDOS:
            self.registrar("ACTIVO: " + self.etapa)
            self.mostrar(forzar=True)

    def fit_terminado(self, msg):
        self.completados += 1
        self.registrar(msg)
        self.mostrar(etapa=msg, forzar=True)

    def finalizar(self, msg):
        """Etapa terminada: panel verde al 100 %, mensaje en la celda y aviso sonoro."""
        self.registrar(f"FINALIZADO: {msg} ({self.completados}/{self.total} contados)")
        self.estado = "terminado"
        self.completados = max(self.completados, self.total)
        self.mostrar(etapa=msg, forzar=True)
        print(f"\n✅ [{NOMBRE_EXPERIMENTO}] {msg} — {datetime.now():%Y-%m-%d %H:%M:%S}", flush=True)
        avisar_fin(ok=True)

    def fallar(self, msg):
        """Etapa detenida (error o interrupción): panel rojo, mensaje en la celda y aviso sonoro."""
        self.registrar(f"DETENIDO: {msg}")
        self.estado = "error"
        self.mostrar(etapa=msg, forzar=True)
        print(f"\n❌ [{NOMBRE_EXPERIMENTO}] {msg} — {datetime.now():%Y-%m-%d %H:%M:%S}", flush=True)
        avisar_fin(ok=False)


def avisar_fin(ok=True):
    """Pitido de Windows al terminar (3 veces) o al detenerse (1 vez); en otros sistemas no hace nada."""
    if not AVISO_SONORO:
        return
    try:
        import winsound

        for _ in range(3 if ok else 1):
            winsound.MessageBeep(winsound.MB_ICONASTERISK if ok else winsound.MB_ICONHAND)
            time.sleep(0.5)
    except Exception:
        pass

## 3. Integridad de los CSV y validación por grupos

Los CSV provienen de `01_preprocesamiento_ML.ipynb`. Ese notebook, dentro del train de cada variante, retira todas las apariciones de textos con etiquetas contradictorias y deja una sola copia de los textos repetidos con la misma etiqueta (coincidencia exacta de `MESSAGE`). El test oficial **no** se usa para eliminar registros del train; las coincidencias train ↔ test no se eliminan y se reportan en `05_comparacion_final_test.ipynb` como análisis de sensibilidad.

Este notebook tampoco compara ni elimina filas contra el test oficial.

Aquí se verifica únicamente:

- alineación entre `normal`, `stem` y `lemma` (mismo número de filas, mismas etiquetas y mismas columnas `ID`/`MESSAGE`);
- rasgos lingüísticos idénticos entre las tres representaciones;
- construcción de un identificador de grupo por fila (se guarda en `GRUPOS_TRAIN`, no como columna del CSV): dos filas quedan en el mismo grupo si coinciden, tras `casefold()` y `strip()`, en el texto `normal`, `stem` o `lemma`, o en el `MESSAGE` original; la unión es transitiva (componentes conexos). Las etiquetas no intervienen en la agrupación;
- ausencia de grupos compartidos entre train y validación en los splits Outer, Inner y en los de la búsqueda final.

Los grupos con etiquetas contradictorias se conservan sin modificar y se exportan a `auditoria_etiquetas_contradictorias.csv`. `StratifiedGroupKFold` mantiene cada grupo completo dentro de un solo lado del split.

In [ ]:
# ============================================================
# 3. CARGA, INTEGRIDAD Y GRUPOS SIN FUGA ENTRE FOLDS
# ============================================================
GRUPOS_TRAIN = {}  # Variante -> np.array; solo contiene información del TRAIN
AUDITORIA_GRUPOS = []
AUDITORIA_CONFLICTOS = []
AUDITORIA_PARTICIONES = []


class UnionFind:
    """Componentes conexos: unión por rango con compresión de caminos."""

    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def cargar_split(variante, prep, split="train"):
    """Carga el CSV preprocesado de una variante, representación y split."""
    if variante not in VARIANTES or prep not in PREPROCESAMIENTOS:
        raise ValueError(f"Variante/preprocessing no reconocido: {variante}/{prep}")
    ruta = DATA_DIR / f"{split}_clean{PREPROCESAMIENTOS[prep]}_{variante}.csv"
    if not Path(ruta).is_file():
        raise FileNotFoundError(f"Falta el CSV: {ruta}")
    # Preservar los IDs originales: no forzarles una conversión numérica.
    return pd.read_csv(ruta, dtype={"ID": "string"}).reset_index(drop=True)


def preparar_xy(df):
    """Separa texto + 9 rasgos (X) y etiqueta (y)."""
    columnas = ["MESSAGE_CLEAN"] + FEATURE_COLS
    faltantes = [c for c in columnas + ["IS_IRONIC"] if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")
    X = df[columnas].copy()
    y = df["IS_IRONIC"].astype(int).reset_index(drop=True)
    X["MESSAGE_CLEAN"] = X["MESSAGE_CLEAN"].fillna("").astype(str)
    X[FEATURE_COLS] = X[FEATURE_COLS].fillna(0)
    return X.reset_index(drop=True), y


def texto_clave(serie):
    """Normalización usada solo para comparar textos al agrupar (no altera el texto del modelo)."""
    # Misma normalización de COMPARACIÓN que la auditoría previa; no modifica el texto del modelo.
    return serie.fillna("").astype(str).str.casefold().str.strip()


def construir_grupos(variante, dfs_por_prep, y):
    """Componentes conexos: agrupa un mismo texto tras normal, stem O lemma.
    No deduplica, no usa las etiquetas para construir grupos y conserva rasgos originales.
    Añade coincidencias de MESSAGE original exacto, si está disponible.
    """
    n = len(y)
    uf = UnionFind(n)

    for prep, df in dfs_por_prep.items():
        primero = {}
        for i, clave in enumerate(texto_clave(df["MESSAGE_CLEAN"])):
            if not clave:
                raise ValueError(
                    f"{variante}/{prep}: texto limpio vacío en fila {i}; revisar antes de agrupar"
                )
            if clave in primero:
                uf.union(primero[clave], i)
            else:
                primero[clave] = i
    # Si dos textos originales son idénticos, deben permanecer en el mismo grupo.
    df_ref = dfs_por_prep["normal"]
    if "MESSAGE" in df_ref:
        primero_raw = {}
        for i, clave in enumerate(texto_clave(df_ref["MESSAGE"])):
            if not clave:
                continue
            if clave in primero_raw:
                uf.union(primero_raw[clave], i)
            else:
                primero_raw[clave] = i

    miembros = {}
    for i in range(n):
        miembros.setdefault(uf.find(i), []).append(i)
    # Id de grupo basado en el menor índice original; determinista entre ambos notebooks.
    ids = {ra: min(idx) for ra, idx in miembros.items()}
    grupos = np.array([ids[uf.find(i)] for i in range(n)], dtype=int)
    duplicados = {ids[ra]: idx for ra, idx in miembros.items() if len(idx) > 1}
    conflictivos = [(g, idx) for g, idx in duplicados.items() if y.iloc[idx].nunique() > 1]
    if conflictivos:
        warnings.warn(
            f"{variante.upper()}: {len(conflictivos)} grupo(s) con etiquetas contradictorias. "
            "Se conservan las etiquetas originales y se mantienen juntos todos sus registros. "
            "Revisar manualmente antes de interpretar resultados.",
            stacklevel=2,
        )
    for g, idx in conflictivos:
        for i in idx:
            AUDITORIA_CONFLICTOS.append(
                {
                    "variante": variante,
                    "grupo": int(g),
                    "fila_0base": int(i),
                    "id_original": str(df_ref.iloc[i]["ID"]) if "ID" in df_ref else "",
                    "etiqueta": int(y.iloc[i]),
                    "texto_original": str(df_ref.iloc[i]["MESSAGE"]) if "MESSAGE" in df_ref else "",
                    "texto_normal": str(df_ref.iloc[i]["MESSAGE_CLEAN"]),
                }
            )
    AUDITORIA_GRUPOS.append(
        {
            "variante": variante,
            "n_train": n,
            "n_grupos": len(miembros),
            "n_grupos_multiples": len(duplicados),
            "n_filas_en_grupos_multiples": sum(map(len, duplicados.values())),
            "tamano_grupo_max": max(map(len, miembros.values())),
            "n_grupos_etiquetas_contradictorias": len(conflictivos),
            "criterio": "union_texto_normal_stem_lemma_y_original_exacto",
        }
    )
    return grupos


def cargar_todos_preps(variante, split="train"):
    """Carga normal/stem/lemma de una variante, verifica que estén alineados y construye los grupos."""
    dfs = {p: cargar_split(variante, p, split=split) for p in PREPROCESAMIENTOS}
    ref = dfs["normal"]
    X_ref, y_ref = preparar_xy(ref)
    datos = {"normal": X_ref}
    for prep, df in dfs.items():
        if prep == "normal":
            continue
        X, y = preparar_xy(df)
        if len(df) != len(ref) or not y.equals(y_ref):
            raise ValueError(f"{variante}/{prep}: diferente número de filas o etiquetas")
        for c in ["ID", "MESSAGE"]:
            if c in ref.columns and c not in df.columns:
                raise ValueError(f"{variante}/{prep}: falta columna de alineación {c}")
            if c in ref.columns and not ref[c].astype(str).equals(df[c].astype(str)):
                raise ValueError(f"{variante}/{prep}: columna {c} desalineada")
        if not X[FEATURE_COLS].equals(X_ref[FEATURE_COLS]):
            raise ValueError(f"{variante}/{prep}: rasgos lingüísticos no alineados")
        datos[prep] = X
    if split == "train" and variante not in GRUPOS_TRAIN:
        GRUPOS_TRAIN[variante] = construir_grupos(variante, dfs, y_ref)
    return datos, y_ref


def grupos_train(variante):
    """Devuelve los grupos del train de una variante (los construye si aún no existen)."""
    if variante not in GRUPOS_TRAIN:
        cargar_todos_preps(variante, split="train")
    return GRUPOS_TRAIN[variante]


def verificar_particion(grupos, a, b, variante, nivel, outer, inner=None):
    """Comprueba que train y validación no compartan grupos y registra la partición."""
    solapamiento = set(grupos[a]) & set(grupos[b])
    if solapamiento:
        raise AssertionError(
            f"FUGA: {variante}/{nivel}/{outer}/{inner} grupos compartidos: {solapamiento}"
        )
    AUDITORIA_PARTICIONES.append(
        {
            "variante": variante,
            "nivel": nivel,
            "outer_fold": outer,
            "inner_fold": inner,
            "n_train": len(a),
            "n_validacion": len(b),
            "n_grupos_compartidos": 0,
        }
    )


def crear_splits_outer(y, grupos, variante="?"):
    """Outer CV agrupada y estratificada."""
    cv = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "outer", fold)
    return splits


def crear_splits_inner(y, grupos, outer_fold, variante="?"):
    """Inner CV agrupada dentro de un Outer Train (semilla RANDOM_STATE + outer_fold)."""
    cv = StratifiedGroupKFold(
        n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE + outer_fold
    )
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "inner", outer_fold, fold)
    return splits


def crear_splits_final(y, grupos, variante="?"):
    """CV agrupada sobre todo el train para la búsqueda final."""
    cv = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "final", 0, fold)
    return splits


# ============================================================
# AUDITORÍA PREVIA — SOLO TRAIN Y FOLDS
# ============================================================
for v in VARIANTES:
    _, etiquetas = cargar_todos_preps(v, split="train")
    gv = grupos_train(v)

    for o, (a, b) in enumerate(crear_splits_outer(etiquetas, gv, v), 1):
        crear_splits_inner(etiquetas.iloc[a].reset_index(drop=True), gv[a], o, v)

    crear_splits_final(etiquetas, gv, v)


df_auditoria_grupos = pd.DataFrame(AUDITORIA_GRUPOS)

df_auditoria_particiones = pd.DataFrame(AUDITORIA_PARTICIONES).drop_duplicates()

df_auditoria_conflictos = pd.DataFrame(AUDITORIA_CONFLICTOS)

print("[AUDIT][ML] Grupos del train")
display(df_auditoria_grupos)

print("\n[AUDIT][ML] Grupos compartidos entre train y validación (todos deben ser 0):")
display(
    df_auditoria_particiones.groupby(["variante", "nivel"], as_index=False)[
        "n_grupos_compartidos"
    ].max()
)

pd.DataFrame(AUDITORIA_GRUPOS).to_csv(OUT_DIR / "auditoria_grupos.csv", index=False)

df_auditoria_particiones.to_csv(OUT_DIR / "auditoria_folds.csv", index=False)

df_auditoria_conflictos.to_csv(OUT_DIR / "auditoria_etiquetas_contradictorias.csv", index=False)

print("\nPRE-FLIGHT DE DATOS: OK")

## 4. TF-IDF original LR/RF, común a LR, RF y XGBoost

Se reproduce `TfidfVectorizer(stop_words=STOP_WORDS, max_features=10000)` del notebook original de LR/RF (resto de parámetros por defecto: `lowercase=True`, `min_df=1`, `max_df=1.0`, `sublinear_tf=False`), **incluyendo** la comparación `(1,1)` vs `(1,2)` dentro del Inner CV. A la matriz TF-IDF se le concatenan los nueve rasgos lingüísticos mediante `passthrough`, es decir, **sin escalar**. Todos los algoritmos tienen las mismas opciones de representación; cada uno puede elegir un rango de n-gramas diferente sin afectar la evaluación independiente del Outer Fold.

**Nota sobre stopwords:** se usa la lista NLTK de español (313 palabras) tal cual, para este experimento controlado. La lista incluye términos de negación e intensidad que pueden ser útiles en sarcasmo (por ejemplo «no», «ni», «sin», «muy», «nada»). La misma lista, en formas completas, se aplica a las tres versiones del texto, incluidas `stem` y `lemma`. Si posteriormente se evalúa otra lista, tendrá que ser otra corrida separada.

In [ ]:
def crear_preprocesador_tfidf(ngram_range=(1, 1)):
    """Exactamente el TF-IDF base original LR/RF y sus 9 rasgos."""
    tfidf = TfidfVectorizer(stop_words=STOP_WORDS, max_features=10000, ngram_range=ngram_range)
    return ColumnTransformer(
        transformers=[
            ("tfidf", tfidf, "MESSAGE_CLEAN"),
            ("linguisticas", "passthrough", FEATURE_COLS),
        ],
        remainder="drop",
    )


# Única representación compartida por todos los modelos.
crear_preprocesador_lr_rf = crear_preprocesador_tfidf
crear_preprocesador_xgb = crear_preprocesador_tfidf

print(
    "[CONFIG][ML] TF-IDF: stopwords NLTK, max_features=10000, n-gramas (1,1)/(1,2) elegidos en el Inner CV."
)

## 5. Pipelines de LR y RF (mismos de la v1)

In [ ]:
def crear_pipeline_lr_rf(modelo):
    """Pipeline TF-IDF + 9 rasgos + LR o RF (class_weight='balanced')."""
    if modelo == "LR":
        clf = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE, class_weight="balanced")
    elif modelo == "RF":
        clf = RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced", n_jobs=1)
    else:
        raise ValueError(f"Modelo LR/RF no reconocido: {modelo}")
    return Pipeline(steps=[("features", crear_preprocesador_lr_rf()), ("clf", clf)])


def grid_lr_rf(modelo):
    """Rejilla de GridSearchCV (incluye el rango de n-gramas)."""
    if modelo == "LR":
        grid = {"clf__C": [0.1, 1.0, 10], "clf__solver": ["liblinear"]}
    elif modelo == "RF":
        grid = {
            "clf__n_estimators": [100, 200, 300],
            "clf__max_depth": [None, 10, 20],
            "clf__min_samples_split": [2, 5],
        }
    else:
        raise ValueError(modelo)
    grid["features__tfidf__ngram_range"] = [(1, 1), (1, 2)]
    return grid


def buscar_lr_rf_train(
    X_por_prep, y, modelo, inner_splits, monitor=None, variante=None, outer_fold=None
):
    """Selecciona preprocessing + hiperparámetros EXCLUSIVAMENTE por puntuación Inner CV."""
    mejor = None
    candidatos = []
    for prep, X in X_por_prep.items():
        fase = f"Outer {outer_fold}/{N_OUTER}" if outer_fold is not None else "Final"
        etiqueta = f"[{str(variante).upper()}][{modelo}][{fase}] GridSearch | prep={prep}"
        if monitor is not None:
            monitor.iniciar(etiqueta)
        search = GridSearchCV(
            estimator=crear_pipeline_lr_rf(modelo),
            param_grid=grid_lr_rf(modelo),
            cv=inner_splits,
            scoring="f1_macro",
            n_jobs=N_JOBS_GRID,
            refit=True,
            error_score="raise",
        )
        search.fit(X, y)
        if monitor is not None:
            monitor.fit_terminado(etiqueta + " | completado")
        scores_folds = [
            search.cv_results_[f"split{i}_test_score"][search.best_index_]
            for i in range(len(inner_splits))
        ]
        desviacion = float(np.std(scores_folds, ddof=1))
        candidato = {
            "prep": prep,
            "inner_f1": float(search.best_score_),
            "inner_std": desviacion,
            "params": search.best_params_,
            "estimator": search.best_estimator_,
        }
        candidatos.append({k: v for k, v in candidato.items() if k != "estimator"})
        if (
            mejor is None
            or candidato["inner_f1"] > mejor["inner_f1"]
            or (
                np.isclose(candidato["inner_f1"], mejor["inner_f1"])
                and candidato["inner_std"] < mejor["inner_std"]
            )
        ):
            mejor = candidato
    return mejor, candidatos


def nested_cv_lr_rf_variante(variante, modelo, monitor=None):
    """Nested CV agrupada de LR o RF para una variante."""
    X_por_prep, y = cargar_todos_preps(variante, split="train")
    detalle, candidatos_inner = [], []
    grupos = grupos_train(variante)
    for outer_fold, (i_train, i_test) in enumerate(
        crear_splits_outer(y, grupos, variante), start=1
    ):
        y_train, y_test = y.iloc[i_train].reset_index(drop=True), y.iloc[i_test].reset_index(
            drop=True
        )
        X_train = {p: X.iloc[i_train].reset_index(drop=True) for p, X in X_por_prep.items()}
        X_outer = {p: X.iloc[i_test].reset_index(drop=True) for p, X in X_por_prep.items()}
        mejor, candidatos = buscar_lr_rf_train(
            X_train,
            y_train,
            modelo,
            crear_splits_inner(y_train, grupos[i_train], outer_fold, variante),
            monitor=monitor,
            variante=variante,
            outer_fold=outer_fold,
        )
        for c in candidatos:
            candidatos_inner.append(
                {
                    "variante": variante,
                    "modelo": modelo,
                    "outer_fold": outer_fold,
                    "preprocessing": c["prep"],
                    "inner_f1_macro": c["inner_f1"],
                    "inner_f1_std": c["inner_std"],
                    "best_params": c["params"],
                }
            )
        prep = mejor["prep"]
        # search.best_estimator_ ya es un refit nuevo sobre TODO este outer_train.
        estimador = mejor["estimator"]
        pred_train = estimador.predict(X_train[prep])
        pred_outer = estimador.predict(X_outer[prep])
        train_f1 = f1_score(y_train, pred_train, average="macro")
        outer_f1 = f1_score(y_test, pred_outer, average="macro")
        # Cada registro figura exactamente una vez como Outer Test en una CV 5-fold.
        OOF_REGISTROS.append(
            pd.DataFrame(
                {
                    "variante": variante,
                    "modelo": modelo,
                    "outer_fold": outer_fold,
                    "idx_train_oficial": i_test,
                    "y_true": y_test.to_numpy(),
                    "y_pred": np.asarray(pred_outer),
                }
            )
        )
        f1_clase = f1_score(y_test, pred_outer, labels=[0, 1], average=None, zero_division=0)
        detalle.append(
            {
                "variante": variante,
                "modelo": modelo,
                "outer_fold": outer_fold,
                "outer_f1_clase_0": float(f1_clase[0]),
                "outer_f1_clase_1": float(f1_clase[1]),
                "best_preprocessing": prep,
                "train_f1_macro": float(train_f1),
                "inner_best_f1": mejor["inner_f1"],
                "inner_best_std": mejor["inner_std"],
                "outer_f1_macro": float(outer_f1),
                "generalization_gap": float(train_f1 - outer_f1),
                "best_params": mejor["params"],
            }
        )
        print(
            f"[{variante.upper()}][{modelo}][Outer {outer_fold}/{N_OUTER}] FINALIZADO | "
            f"prep={prep} | F1_inner={mejor['inner_f1']:.4f}±{mejor['inner_std']:.4f} | "
            f"F1_train={train_f1:.4f} | F1_outer={outer_f1:.4f} | gap={train_f1 - outer_f1:.4f}"
        )
    return pd.DataFrame(detalle), pd.DataFrame(candidatos_inner)

## 6. Pipeline de XGBoost (mismo de la v1)

In [ ]:
# Alias compartido: mismo TF-IDF original LR/RF para LR, RF y XGBoost.
crear_preprocesador = crear_preprocesador_tfidf

# ============================================================
# 6. ESPACIO DE BÚSQUEDA PARA OPTUNA (XGBoost)
# ============================================================


def sugerir_params_xgb(trial):
    """Espacio de búsqueda de XGBoost para Optuna."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 700, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 15),
        "subsample": trial.suggest_float("subsample", 0.55, 0.90),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.90),
        "gamma": trial.suggest_float("gamma", 0.5, 8.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 2.5),
    }


def crear_xgboost(params, random_state=RANDOM_STATE):
    """Clasificador XGBoost con los parámetros fijos del experimento."""
    return XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=N_JOBS_XGB,
        **params,
    )


def crear_pipeline(params, random_state=RANDOM_STATE, ngramas="uni"):
    """Pipeline TF-IDF + 9 rasgos + XGBoost con el rango de n-gramas indicado."""
    if ngramas not in NGRAMAS_OPCIONES:
        raise ValueError(f"Opción de n-gramas no reconocida: {ngramas}")
    return Pipeline(
        steps=[
            ("features", crear_preprocesador(ngram_range=NGRAMAS_OPCIONES[ngramas])),
            ("xgb", crear_xgboost(params, random_state=random_state)),
        ]
    )


def avance_optuna(monitor, etiqueta, n_trials):
    """Callback de Optuna: actualiza el panel de progreso al terminar cada trial."""

    def callback(study, trial):
        if monitor is None:
            return
        valor = f"{trial.value:.4f}" if trial.value is not None else "—"
        monitor.fit_terminado(
            f"{etiqueta} | trial {len(study.trials)}/{n_trials} | F1={valor} | mejor={study.best_value:.4f}"
        )

    return callback

## 7. Configuración final de la v1 (no se repite la búsqueda)

Se lee `seleccion_final_ml.csv` de la v1: un modelo por variante y algoritmo, con su preprocesamiento (`normal/stem/lemma`), n-gramas e hiperparámetros elegidos con la búsqueda final sobre todo el train. También se lee el resumen de la Nested CV de la v1 como referencia.

In [ ]:
ruta_sel = OUT_DIR_V1 / "seleccion_final_ml.csv"
if not ruta_sel.exists():
    raise FileNotFoundError(f"Falta {ruta_sel}: ejecuta antes la v1 (búsqueda final) o copia su carpeta de resultados.")
SELECCION = pd.read_csv(ruta_sel)
SELECCION["best_params"] = SELECCION["best_params"].map(ast.literal_eval)
SELECCION = SELECCION[SELECCION["variante"].isin(VARIANTES)].sort_values(["variante", "modelo"]).reset_index(drop=True)
assert len(SELECCION) == len(VARIANTES) * len(MODELOS_ML), "Se esperan 3 modelos por variante en la selección final"
display(SELECCION[["variante", "modelo", "preprocessing", "ngramas", "best_f1_cv_final", "best_params"]])
ruta_nested_v1 = OUT_DIR_V1 / "resumen_nested_ml.csv"
if ruta_nested_v1.exists():
    print("Nested CV de la v1 (referencia, no cambia):")
    display(pd.read_csv(ruta_nested_v1).round(4))

## 8. Entrenamiento de los modelos finales con 5 semillas (solo train)

Para cada variante, algoritmo y semilla se arma el **mismo pipeline** de la v1 (`crear_pipeline_lr_rf` para LR/RF; `crear_pipeline` para XGBoost) con los hiperparámetros de la selección final, se cambia **solo** la semilla (`random_state`) y se entrena con todo el train en el preprocesamiento elegido. Los modelos se guardan en `modelos_finales/{lr|rf|xgboost}_{variante}_{preprocessing}_seed{s}.pkl`; si ya existen, se reutilizan.

- **LR:** `random_state` no cambia el resultado de `liblinear` con estos datos (se verifica en la sección 9).
- **RF:** la semilla cambia el muestreo bootstrap y las variables candidatas de cada árbol.
- **XGBoost:** la semilla cambia el submuestreo de filas (`subsample`) y columnas (`colsample_bytree`).

Para acelerar, RF usa todos los núcleos (`n_jobs=-1`); con `random_state` fijo esto no cambia el resultado.

In [ ]:
def construir_modelo(fila, seed):
    modelo, params = fila["modelo"], dict(fila["best_params"])
    if modelo in ("LR", "RF"):
        pipe = crear_pipeline_lr_rf(modelo)
        pipe.set_params(**params)
        pipe.set_params(clf__random_state=seed)
        if modelo == "RF":
            pipe.set_params(clf__n_jobs=-1)
        return pipe
    return crear_pipeline(params, random_state=seed, ngramas=fila["ngramas"])


def ruta_modelo(fila, seed):
    return MODELS_DIR / f"{fila['modelo'].lower()}_{fila['variante']}_{fila['preprocessing']}_seed{seed}.pkl"


MODELOS_SEMILLA = {}
monitor_sem = Monitor(
    OUT_DIR,
    total=len(SELECCION) * len(SEMILLAS_FINALES),
    ya_completados=sum(ruta_modelo(f, s).exists() for _, f in SELECCION.iterrows() for s in SEMILLAS_FINALES),
    unidad="Modelos finales entrenados",
)
try:
    for variante in VARIANTES:
        X_por_prep, y = cargar_todos_preps(variante, split="train")
        for _, fila in SELECCION[SELECCION["variante"] == variante].iterrows():
            for seed in SEMILLAS_FINALES:
                ruta = ruta_modelo(fila, seed)
                clave = (variante, fila["modelo"], seed)
                if ruta.exists():
                    MODELOS_SEMILLA[clave] = joblib.load(ruta)
                    continue
                monitor_sem.iniciar(f"[{variante.upper()}][{fila['modelo']}] semilla {seed}")
                estimador = construir_modelo(fila, seed)
                estimador.fit(X_por_prep[fila["preprocessing"]], y)
                joblib.dump(estimador, ruta)
                MODELOS_SEMILLA[clave] = estimador
                monitor_sem.fit_terminado(f"[{variante.upper()}][{fila['modelo']}] semilla {seed} entrenada")
except KeyboardInterrupt:
    monitor_sem.fallar("Entrenamiento interrumpido; los modelos guardados se reutilizan al reejecutar")
    raise
except Exception as exc:
    monitor_sem.registrar(f"ERROR {type(exc).__name__}: {exc}")
    monitor_sem.fallar(f"Entrenamiento con error {type(exc).__name__}; consultar monitor.log")
    raise
monitor_sem.finalizar("Modelos finales con 5 semillas entrenados (solo train)")
print("Modelos disponibles:", len(MODELOS_SEMILLA))

## 9. Test oficial con 5 semillas — bloqueado

Se habilita con `EJECUTAR_TEST_OFICIAL=True` y `CONFIRMACION_TEST="TEST_OFICIAL_CONGELADO"`. Para cada modelo y semilla se predice el test oficial; se reportan F1-Macro, F1 de la clase irónica y F1-Macro **sin** los textos que coinciden con el train (mismo conjunto común que la comparación final), con media ± desviación entre semillas. No se elige la mejor semilla ni se cambia ningún hiperparámetro.

**Verificaciones:** (1) la semilla 42 reproduce las predicciones de la v1 (`predicciones_test_ml.csv`); (2) la desviación de LR entre semillas es 0 (modelo determinista).

In [ ]:
def ids_coincidentes(variante):
    """Conjunto común de textos del test que coinciden con el train (de la comparación final v6, si existe)."""
    ruta = DATA_DIR / "resultados_comparacion_final_v6" / "ids_test_coincidentes_con_train.csv"
    if ruta.exists():
        d = pd.read_csv(ruta, dtype={"ID": str})
        return set(d.loc[d["variante"] == variante, "ID"].astype(str))
    tr, te = cargar_split(variante, "normal", "train"), cargar_split(variante, "normal", "test")
    return set(te.loc[te["MESSAGE"].isin(set(tr["MESSAGE"])), "ID"].astype(str))


if not EJECUTAR_TEST_OFICIAL:
    print("TEST OFICIAL BLOQUEADO: los modelos ya están entrenados solo con el train.")
else:
    if CONFIRMACION_TEST != "TEST_OFICIAL_CONGELADO":
        raise ValueError("Bloqueado: confirma explícitamente la evaluación final con CONFIRMACION_TEST.")
    filas, predicciones = [], []
    for variante in VARIANTES:
        coincidentes = ids_coincidentes(variante)
        for _, fila in SELECCION[SELECCION["variante"] == variante].iterrows():
            df_test = cargar_split(variante, fila["preprocessing"], split="test")
            X_test, y_test = preparar_xy(df_test)
            ids = df_test["ID"].astype(str).to_numpy()
            limpio = ~np.isin(ids, list(coincidentes))
            for seed in SEMILLAS_FINALES:
                est = MODELOS_SEMILLA[(variante, fila["modelo"], seed)]
                pred = np.asarray(est.predict(X_test), dtype=int)
                prob = np.asarray(est.predict_proba(X_test)[:, 1], dtype=float)
                predicciones.append(pd.DataFrame({"variante": variante, "modelo": fila["modelo"], "seed": seed,
                                                  "ID": ids, "y_true": y_test.to_numpy(), "y_pred": pred, "prob_ironia": prob}))
                filas.append({
                    "variante": variante, "modelo": fila["modelo"], "seed": seed, "preprocessing": fila["preprocessing"],
                    "f1_macro": f1_score(y_test, pred, average="macro"),
                    "f1_ironia": f1_score(y_test, pred, pos_label=1, zero_division=0),
                    "f1_macro_sin_coincidencias": f1_score(y_test[limpio], pred[limpio], average="macro"),
                    "accuracy": accuracy_score(y_test, pred),
                    "n_coincidencias": int((~limpio).sum()),
                })
    RES_SEMILLAS = pd.DataFrame(filas)
    PRED_SEMILLAS = pd.concat(predicciones, ignore_index=True)
    RES_SEMILLAS.to_csv(OUT_DIR / "resultados_test_ml_5_semillas.csv", index=False)
    PRED_SEMILLAS.to_csv(OUT_DIR / "predicciones_test_ml_5_semillas.csv", index=False)
    RESUMEN = (RES_SEMILLAS.groupby(["variante", "modelo"])
               .agg(f1_macro_mean=("f1_macro", "mean"), f1_macro_std=("f1_macro", "std"),
                    f1_ironia_mean=("f1_ironia", "mean"), f1_ironia_std=("f1_ironia", "std"),
                    f1_sin_coinc_mean=("f1_macro_sin_coincidencias", "mean"),
                    f1_sin_coinc_std=("f1_macro_sin_coincidencias", "std"))
               .reset_index())
    RESUMEN.to_csv(OUT_DIR / "resumen_test_ml_5_semillas.csv", index=False)
    print("F1-Macro por semilla:")
    display(RES_SEMILLAS.pivot_table(index=["variante", "modelo"], columns="seed", values="f1_macro").round(4))
    print("Media ± desviación entre semillas:")
    display(RESUMEN.round(4))

    # --- verificaciones ---
    ruta_v1 = OUT_DIR_V1 / "predicciones_test_ml.csv"
    if ruta_v1.exists():
        v1p = pd.read_csv(ruta_v1, dtype={"ID": str}).set_index(["variante", "modelo", "ID"])
        s42 = PRED_SEMILLAS[PRED_SEMILLAS["seed"] == 42].set_index(["variante", "modelo", "ID"])
        comun = s42.join(v1p[["y_pred"]], rsuffix="_v1", how="inner")
        acuerdo = (comun["y_pred"] == comun["y_pred_v1"]).groupby(level=[0, 1]).mean()
        print("Acuerdo de la semilla 42 con la v1 (debe ser 1.0):")
        display(acuerdo.rename("acuerdo").reset_index())
    print("Desviación de LR entre semillas (debe ser 0):",
          RESUMEN.loc[RESUMEN["modelo"] == "LR", ["variante", "f1_macro_std"]].to_dict("records"))

## 10. Interpretabilidad con SHAP (modelos de la semilla 42)

Se explican los modelos de la semilla 42 (los mismos de la v1) sobre los 600 tuits del test de cada variante. **No se usan las etiquetas** y nada de esto cambia los modelos. Cada característica es una palabra (o n-grama) del TF-IDF —en el preprocesamiento elegido para ese modelo, por lo que puede ser una raíz (`stem`) o un lema (`lemma`)— o uno de los 9 rasgos lingüísticos.

- **LR:** contribución exacta de un modelo lineal, `coef × (x − media del train)` en log-odds (equivale a SHAP lineal con variables independientes).
- **RF y XGBoost:** `shap.TreeExplainer` (valores exactos para árboles; RF en probabilidad, XGBoost en log-odds).

Para palabras y rasgos se promedia la contribución **en los tuits donde aparecen** (así el signo indica hacia dónde empuja su presencia). Un valor **positivo** empuja hacia *irónico*. Como las escalas difieren entre modelos (log-odds vs. probabilidad), se compara el **orden** de las palabras, no la magnitud. Solo entran al análisis global las palabras presentes en ≥ `MIN_TEXTOS_PALABRA` tuits del test.

**Salidas:** `ml_v2_shap_global.csv`, `ml_v2_shap_rasgos.csv`, `ml_v2_shap_palabras.png/.pdf` y `ml_v2_shap_rasgos.png/.pdf`.

In [ ]:
try:
    import shap
except ImportError as exc:
    raise ImportError("Falta el paquete shap. Instálalo con:  py -m uv add shap") from exc
import scipy.sparse as sp

NOMBRES_RASGOS = {"n_exc": "exclamaciones", "n_int": "interrogaciones", "n_may": "mayúsculas", "n_emo": "emojis",
                  "n_ris": "risas", "n_neg": "negaciones", "n_elo": "alargamientos", "n_com": "comillas",
                  "n_pun": "puntos suspensivos"}


def matriz(preproc, X):
    M = preproc.transform(X)
    return sp.csr_matrix(M, dtype=np.float64) if not sp.issparse(M) else M.tocsr().astype(np.float64)


def contribuciones(modelo, estimador, X_train, X_test):
    preproc = estimador.named_steps["features"]
    clf = estimador.named_steps["xgb" if modelo == "XGBoost" else "clf"]
    nombres = preproc.get_feature_names_out()
    Mte = matriz(preproc, X_test)
    if modelo == "LR":
        media = np.asarray(matriz(preproc, X_train).mean(axis=0)).ravel()
        coef = clf.coef_.ravel()
        valores = Mte.toarray() * coef - media * coef
    else:
        explicador = shap.TreeExplainer(clf)
        sv = explicador.shap_values(Mte.toarray() if modelo == "RF" else Mte, check_additivity=False)
        sv = np.asarray(sv[1] if isinstance(sv, list) else sv)
        valores = sv[:, :, 1] if sv.ndim == 3 else sv
    presente = (Mte != 0).toarray()
    return nombres, valores, presente


filas, filas_rasgos = [], []
for variante in VARIANTES:
    X_tr_por_prep, _ = cargar_todos_preps(variante, split="train")
    for _, fila in SELECCION[SELECCION["variante"] == variante].iterrows():
        modelo, prep = fila["modelo"], fila["preprocessing"]
        est = MODELOS_SEMILLA[(variante, modelo, 42)]
        X_test, _ = preparar_xy(cargar_split(variante, prep, split="test"))
        nombres, valores, presente = contribuciones(modelo, est, X_tr_por_prep[prep], X_test)
        for j, nombre in enumerate(nombres):
            tipo, termino = nombre.split("__", 1)
            if tipo == "linguisticas":
                v = valores[presente[:, j], j]  # tuits donde el rasgo aparece (valor > 0)
                filas_rasgos.append({"variante": variante, "modelo": modelo, "rasgo": NOMBRES_RASGOS.get(termino, termino),
                                     "n_textos": int(presente[:, j].sum()),
                                     "shap_medio": float(v.mean()) if len(v) else np.nan,
                                     "shap_abs_medio": float(np.abs(v).mean()) if len(v) else np.nan})
                continue
            n = int(presente[:, j].sum())
            if n < MIN_TEXTOS_PALABRA:
                continue
            v = valores[presente[:, j], j]  # contribución en los tuits donde aparece la palabra
            filas.append({"variante": variante, "modelo": modelo, "preprocessing": prep, "palabra": termino,
                          "n_textos": n, "shap_medio": float(v.mean()), "shap_abs_medio": float(np.abs(v).mean())})
        print(f"[{variante.upper()}][{modelo}] SHAP calculado ({prep})")

SHAP_ML = pd.DataFrame(filas)
SHAP_RASGOS = pd.DataFrame(filas_rasgos)
SHAP_ML.to_csv(OUT_DIR / "ml_v2_shap_global.csv", index=False)
SHAP_RASGOS.to_csv(OUT_DIR / "ml_v2_shap_rasgos.csv", index=False)

fig, axes = plt.subplots(len(MODELOS_ML), len(VARIANTES), figsize=(5.2 * len(VARIANTES), 4.6 * len(MODELOS_ML)), squeeze=False)
for i, modelo in enumerate(MODELOS_ML):
    for j, variante in enumerate(VARIANTES):
        ax = axes[i, j]
        g = SHAP_ML[(SHAP_ML["variante"] == variante) & (SHAP_ML["modelo"] == modelo)]
        sel = pd.concat([g.nlargest(TOP_PALABRAS, "shap_medio"), g.nsmallest(TOP_PALABRAS, "shap_medio")])
        sel = sel.drop_duplicates("palabra").sort_values("shap_medio")
        ax.barh(range(len(sel)), sel["shap_medio"], color=["#c0392b" if x > 0 else "#2e6da4" for x in sel["shap_medio"]])
        ax.set_yticks(range(len(sel)), [f"{p} (n={n})" for p, n in zip(sel["palabra"], sel["n_textos"])], fontsize=7)
        ax.axvline(0, color="black", lw=0.8)
        ax.grid(axis="x", alpha=0.25)
        prep = g["preprocessing"].iloc[0] if len(g) else ""
        ax.set_title(f"{variante.upper()} · {modelo} ({prep})", fontsize=10)
fig.suptitle("ML v2 — palabras que más empujan hacia irónico (rojo) y no irónico (azul), SHAP", fontsize=12)
fig.tight_layout()
fig.savefig(OUT_DIR / "ml_v2_shap_palabras.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "ml_v2_shap_palabras.pdf", bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(OUT_DIR / "ml_v2_shap_palabras.png")))

tabla = SHAP_RASGOS.pivot_table(index="rasgo", columns=["modelo", "variante"], values="shap_medio")
tabla = tabla.reindex(columns=[(m, v) for m in MODELOS_ML for v in VARIANTES])
fig, ax = plt.subplots(figsize=(1.0 * tabla.shape[1] + 3, 0.45 * tabla.shape[0] + 2))
# Normalización por modelo (escalas distintas): se divide por el máximo |valor| de cada modelo.
norm = tabla.copy()
for m in MODELOS_ML:
    cols = [c for c in norm.columns if c[0] == m]
    escala = np.nanmax(np.abs(norm[cols].to_numpy())) or 1.0
    norm[cols] = norm[cols] / escala
im = ax.imshow(norm.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(tabla.shape[1]), [f"{m}\n{v.upper()}" for m, v in tabla.columns], fontsize=8)
ax.set_yticks(range(tabla.shape[0]), tabla.index)
fig.colorbar(im, ax=ax, label="SHAP medio (normalizado por modelo)")
ax.set_title("Rasgos lingüísticos: dirección de su efecto (rojo = hacia irónico)")
fig.tight_layout()
fig.savefig(OUT_DIR / "ml_v2_shap_rasgos.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_DIR / "ml_v2_shap_rasgos.pdf", bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(OUT_DIR / "ml_v2_shap_rasgos.png")))
display(tabla.round(4))

## 11. Resumen metodológico (v2)

1. La v2 **no repite** la Nested CV ni la búsqueda final: la selección de hiperparámetros y los resultados de desarrollo son los de la v1.
2. Cada modelo final se reentrenó con todo el train y **5 semillas** (42, 52, 62, 72, 82), las mismas de los Transformers, sin cambiar sus hiperparámetros; se reporta media ± desviación en el test. La semilla 42 reproduce la v1.
3. LR (`liblinear`) es determinista: sus 5 semillas dan el mismo resultado. RF y XGBoost varían con la semilla.
4. SHAP explica los modelos de la semilla 42 sobre el test sin usar etiquetas; no interviene en ninguna decisión.

**Referencia:** Lundberg, S. M., y Lee, S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS 30*. — Lundberg, S. M., et al. (2020). From local explanations to global understanding with explainable AI for trees. *Nature Machine Intelligence, 2*, 56–67.